# Survival Analysis & Accelerated Failure Time (AFT) for Credit Risk

In institutional credit risk and banking regulation (IFRS 9 and CECL), binary classification models (which predict default within a fixed 12-month window) are insufficient. Financial institutions must calculate **Lifetime Expected Credit Loss (Lifetime ECL)** across the full term of a loan (e.g., 36 to 60 months).

### The Challenge: Right-Censored Data
When observing a loan portfolio at date $t_{\text{obs}}$:
- Some borrowers have already defaulted at month $t_i$. Their exact time-to-event is known: $T = t_i$.
- Many borrowers are currently performing (active) at month $t_{\text{obs}}$. They have *not yet* defaulted, but may default in future months. Their true default time is **right-censored**: $T \in [t_{\text{obs}}, +\infty)$.

Discarding active loans or treating them as "non-defaulters" introduces massive survival bias. XGBoost's native **Accelerated Failure Time (`survival:aft`)** engine natively handles right-censored intervals.

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from scipy.stats import norm, logistic
from sklearn.model_selection import train_test_split

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. The Accelerated Failure Time (AFT) Formulation

In an AFT model, the natural logarithm of the survival time $T$ is modeled as a linear function of the tree ensemble output plus a scaled random noise term:

$$\ln(T) = f(\mathbf{x}) + \sigma Z$$

Where:
- $f(\mathbf{x})$: The XGBoost ensemble prediction (log expected survival time).
- $\sigma$: Scale parameter (learned during training).
- $Z$: A standard random variable whose distribution is specified by `aft_loss_distribution`:
  - `'normal'`: Log-normal survival model ($Z \sim \mathcal{N}(0, 1)$).
  - `'logistic'`: Log-logistic survival model.
  - `'extreme'`: Weibull survival model (Extreme Value distribution).

### Bounding Intervals in XGBoost
XGBoost represents time-to-event as an interval $[y_{\text{lower}}, y_{\text{upper}}]$:
- **Exact Event (Default at month $t$)**: $y_{\text{lower}} = t$, $y_{\text{upper}} = t$.
- **Right-Censored (Active at observation month $t_{\text{obs}}$)**: $y_{\text{lower}} = t_{\text{obs}}$, $y_{\text{upper}} = +\infty$.
- **Left-Censored (Defaulted prior to start month $t_0$)**: $y_{\text{lower}} = 0$, $y_{\text{upper}} = t_0$.

## 2. Generating a Realistic 60-Month Loan Portfolio

Let's simulate a 5-year (60-month) consumer loan book with typical borrower risk factors and right-censoring at observation month $t_{\text{obs}} = 36$.

In [ ]:
n_loans = 12000

# Borrower Characteristics
credit_score = np.random.normal(680, 50, n_loans)
dti = np.random.uniform(0.1, 0.55, n_loans)
revolving_util = np.random.beta(2, 3, n_loans)
loan_amount = np.random.choice([10000, 15000, 25000, 35000], n_loans)
interest_rate = 0.04 + (850 - credit_score) * 0.0003 + dti * 0.08
annual_income = np.random.lognormal(10.9, 0.45, n_loans)

df = pd.DataFrame({
    'credit_score': credit_score,
    'dti': dti,
    'revolving_util': revolving_util,
    'loan_amount': loan_amount,
    'interest_rate': interest_rate,
    'annual_income': annual_income
})

# Latent log survival time (months)
# Higher credit score & income -> longer time to default (higher survival time)
# Higher DTI & interest rate -> accelerated default
log_t_base = 3.8 + 0.004 * (credit_score - 650) - 1.8 * dti - 1.2 * revolving_util - 4.0 * interest_rate
noise = np.random.normal(0, 0.45, n_loans)
true_time_to_default = np.exp(log_t_base + noise) # True continuous survival time in months

# Observation window: 36 months of tracking
obs_time = 36.0

# Default occurs if true time <= 36 months
defaulted = (true_time_to_default <= obs_time).astype(int)
observed_time = np.where(defaulted == 1, true_time_to_default, obs_time)

# Construct Lower and Upper Bounds for XGBoost AFT Engine
y_lower = observed_time.copy()
y_upper = np.where(defaulted == 1, observed_time, np.inf)

print(f"Total Loans Generated: {n_loans}")
print(f"Defaulted Loans (Observed Event): {sum(defaulted)} ({sum(defaulted)/n_loans*100:.1f}%)")
print(f"Active Loans (Right-Censored at month {obs_time}): {n_loans - sum(defaulted)} ({(n_loans - sum(defaulted))/n_loans*100:.1f}%)")

## 3. Formatting `xgb.DMatrix` with Interval Censoring Bounds

XGBoost provides dedicated float info fields `label_lower_bound` and `label_upper_bound`.

In [ ]:
# Split into Train and Test sets
indices = np.arange(n_loans)
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

X_train, X_test = df.iloc[train_idx], df.iloc[test_idx]
y_lower_tr, y_lower_te = y_lower[train_idx], y_lower[test_idx]
y_upper_tr, y_upper_te = y_upper[train_idx], y_upper[test_idx]
def_test = defaulted[test_idx]

# Build DMatrix with interval bounds
dtrain = xgb.DMatrix(X_train)
dtrain.set_float_info('label_lower_bound', y_lower_tr)
dtrain.set_float_info('label_upper_bound', y_upper_tr)

dtest = xgb.DMatrix(X_test)
dtest.set_float_info('label_lower_bound', y_lower_te)
dtest.set_float_info('label_upper_bound', y_upper_te)

print("DMatrix successfully compiled with right-censoring constraints!")

## 4. Training the AFT Model & Distribution Comparison

We train XGBoost with `objective='survival:aft'` and compare different distribution assumptions (`normal`, `logistic`, `extreme`).

In [ ]:
aft_distributions = ['normal', 'logistic', 'extreme']
models = {}

for dist in aft_distributions:
    params = {
        'objective': 'survival:aft',
        'eval_metric': 'aft-nloglik',
        'aft_loss_distribution': dist,
        'aft_loss_distribution_scale': 1.20,
        'tree_method': 'hist',
        'max_depth': 4,
        'learning_rate': 0.08,
        'random_state': 42
    }
    
    bst = xgb.train(params, dtrain, num_boost_round=120, evals=[(dtrain, 'train'), (dtest, 'val')], verbose_eval=False)
    models[dist] = bst
    
    # Predict log survival time or expected survival time
    pred_time = bst.predict(dtest)
    print(f"Distribution: {dist:8s} -> Median Predicted Survival Time: {np.median(pred_time):.1f} months")

## 5. Concordance Index (C-Index) Evaluation

In survival analysis, standard $R^2$ or ROC-AUC does not account for censored durations. The **Concordance Index (C-Index)** evaluates ranking discrimination: among pairs of borrowers where at least one defaulted, did the model assign a lower predicted survival time to the borrower who defaulted earlier?

In [ ]:
def calculate_c_index(time_pred: np.ndarray, time_true: np.ndarray, event: np.ndarray) -> float:
    """
    Computes Harrell's Concordance Index on right-censored data.
    """
    concordant = 0
    permissible = 0
    n = len(time_true)
    
    for i in range(n):
        for j in range(i + 1, n):
            # Pair is permissible if event occurred for i and true_i < true_j
            if event[i] == 1 and time_true[i] < time_true[j]:
                permissible += 1
                if time_pred[i] < time_pred[j]:
                    concordant += 1
                elif time_pred[i] == time_pred[j]:
                    concordant += 0.5
            elif event[j] == 1 and time_true[j] < time_true[i]:
                permissible += 1
                if time_pred[j] < time_pred[i]:
                    concordant += 1
                elif time_pred[j] == time_pred[i]:
                    concordant += 0.5
                    
    return concordant / permissible if permissible > 0 else 0.5

# Sample 400 test samples for fast pairwise C-index evaluation
eval_idx = np.random.choice(len(X_test), 400, replace=False)
preds_norm = models['normal'].predict(xgb.DMatrix(X_test.iloc[eval_idx]))

c_index = calculate_c_index(preds_norm, y_lower_te[eval_idx], def_test[eval_idx])
print(f"=== SURVIVAL MODEL C-INDEX ===")
print(f"XGBoost Normal AFT C-Index: {c_index:.4f} (Benchmark: >0.70 is strong discrimination)")

## 6. Deriving Individual Survival Curves & Lifetime ECL

Given the predicted median survival time $\hat{T}_i$ for borrower $i$, the survival probability at month $t$ under the normal distribution is:

$$S(t | \mathbf{x}_i) = 1 - \Phi\left( \frac{\ln(t) - \ln(\hat{T}_i)}{\sigma} \right)$$

And cumulative probability of default by month $t$ is:
$$\text{Cumulative PD}(t) = 1 - S(t | \mathbf{x}_i)$$

In [ ]:
# Select 3 distinct risk profiles from test set
sample_preds = models['normal'].predict(xgb.DMatrix(X_test))
low_risk_idx = np.argmax(sample_preds) # Longest predicted survival
high_risk_idx = np.argmin(sample_preds) # Shortest predicted survival
med_risk_idx = np.argsort(sample_preds)[len(sample_preds)//2]

months = np.linspace(1, 60, 100)
sigma = 1.20 # distribution scale

def get_survival_curve(pred_median, t_eval, sigma):
    z = (np.log(t_eval) - np.log(pred_median)) / sigma
    return 1 - norm.cdf(z)

s_low = get_survival_curve(sample_preds[low_risk_idx], months, sigma)
s_med = get_survival_curve(sample_preds[med_risk_idx], months, sigma)
s_high = get_survival_curve(sample_preds[high_risk_idx], months, sigma)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Survival Curves (Probability of NOT defaulting)
axes[0].plot(months, s_low, color='seagreen', linewidth=2.5, label=f'Prime Borrower (Pred Median: {sample_preds[low_risk_idx]:.1f}m)')
axes[0].plot(months, s_med, color='navy', linewidth=2.5, label=f'Near-Prime Borrower (Pred Median: {sample_preds[med_risk_idx]:.1f}m)')
axes[0].plot(months, s_high, color='crimson', linewidth=2.5, label=f'Subprime Borrower (Pred Median: {sample_preds[high_risk_idx]:.1f}m)')
axes[0].set_title("Individual Loan Survival Curves S(t | x)")
axes[0].set_xlabel("Loan Tenure (Months)")
axes[0].set_ylabel("Survival Probability P(T > t)")
axes[0].legend()

# Cumulative Default Risk Curves
axes[1].plot(months, (1 - s_low) * 100, color='seagreen', linewidth=2.5, label='Prime Cumulative Default %')
axes[1].plot(months, (1 - s_med) * 100, color='navy', linewidth=2.5, label='Near-Prime Cumulative Default %')
axes[1].plot(months, (1 - s_high) * 100, color='crimson', linewidth=2.5, label='Subprime Cumulative Default %')
axes[1].axvline(12, color='gray', linestyle='--', label='12-Month Regulatory Horizon')
axes[1].axvline(36, color='gray', linestyle=':', label='36-Month Amortization')
axes[1].set_title("Cumulative Lifetime Default Probability (IFRS 9 / CECL)")
axes[1].set_xlabel("Loan Tenure (Months)")
axes[1].set_ylabel("Cumulative Probability of Default (%)")
axes[1].legend()

plt.tight_layout()
plt.show()